## import

In [3]:
import numpy as np 
import pandas as pd
import random 
import os
from typing import Tuple, List
import time

# https://discuss.pytorch.org/t/how-to-enable-torch-use-cuda-dsa/202824/5
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"
# https://discuss.pytorch.org/t/keep-getting-cuda-oom-error-with-pytorch-failing-to-allocate-all-free-memory/133896/10
# https://dev.to/shittu_olumide_/how-can-i-set-maxsplitsizemb-to-avoid-fragmentation-in-pytorch-37h9
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64 "

from tqdm import tqdm
import torch
import copy
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_selector as selector
from sklearn.compose import ColumnTransformer
import torch_geometric
# from torch_geometric import seed_everything
from torch_geometric.data import Data, HeteroData
from torch_geometric.loader import NeighborLoader, ImbalancedSampler
import torch_geometric.transforms as T
import torch_geometric.utils as U
from torch_geometric.utils import coalesce
from torch_geometric.nn import summary, HeteroConv, GATv2Conv, SAGEConv, Linear
import pyg_lib
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score, f1_score, auc, precision_recall_fscore_support, matthews_corrcoef 
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

import tempfile
# edit the line below to specify specify tmp_directory
# tempfile.tempdir = ''

torch.multiprocessing.set_sharing_strategy('file_system')
# https://stackoverflow.com/questions/71642653/how-to-resolve-the-error-runtimeerror-received-0-items-of-ancdata
import resource
rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
resource.setrlimit(resource.RLIMIT_NOFILE, (4096, rlimit[1]))

print(f"numpy.__version__ = {np.__version__}")
print(f"pandas.__version__ = {pd.__version__}")
print(f"torch.__version__ = {torch.__version__}")
print(f"torch_geometric.__version__ = {torch_geometric.__version__}")
print(f"pyg_lib.__version__ = {pyg_lib.__version__}")

numpy.__version__ = 1.26.3
pandas.__version__ = 2.2.1
torch.__version__ = 2.1.2+cu118
torch_geometric.__version__ = 2.5.2
pyg_lib.__version__ = 0.3.1+pt21cu118


## hyperparameters

In [3]:
hyperparameters = {
    'gene_mode': 'degree',
    'node_mode' : 'degree', #'id' or 'degree' for abcm nodes_types
    'edge_features' : False, # True = those are the true edge_weights from my network
    'Heads' : 1, #2
    # 'conv_number' : 2, # 
    # 'conv_type': 'SAGE', # 
    'sage_aggr': 'sum', # 'sum', 'var'
    'sage_norm': True, # False
    'sage_project': False,
    'heteroconv_aggr' : 'sum', # 'cat', 'max'
    'heteroconv_aggr_1' : 'sum', # 'sum', 'cat', 'max'
    'heteroconv_aggr_2' : 'sum', # 'sum', 'cat', 'max'
    'heteroconv_aggr_3' : 'sum', # 'cat', 'max'
    # 'activation': 'relu', # 'relu', 'tanh'
    'hc_2_q' : 1, # hc_2 = int(i/hc_2_q)
    'hc_3_q': 1, # hc_3_q = int(i/hc_3_q)
    # 'size_of_batch': 32,
    'neighbors': 30, #30, 40, 50
    'disjoint_loader' : True, 
    'zero_out_batch_features': True,
    'epochs_to_train' : 500, #
    'min_epochs_to_train': 3,
    'patience': 10,
    'min_delta': 0.005,
    'auto_loop': False,
    'nodes_pairs_features': 'zeros', #
    'dnds_mode': 'sum',
    'type_to_return': 'torch', # 'numpy'
    'final_type': 'torch', # 'numpy'
    'remove_forbidden_value': True,
    'shuffle_train': True,
    'ana_edges': 'common', # 'all
    'string_threshold': 700, # confidence = {'low': 0.15, 'medium': 0.4, 'high': 0.7, 'highest': 0.9} from STRING doc, * 1000 to have EdgeScore
    'setting': 'pairs_to_pairs',
    'global_loader_neighbors': 'all' # neighbors
}

In [3]:
# 15 hours with batch_size = 32, cn = 2, only SAGE, no 128
# 21 hours with batch_size = 16, cn = 2, only SAGE, no 128
## another loop for batch_size ?
## batch_size = 16

# # https://github.com/pyg-team/pytorch_geometric/discussions/8716
# def set_random_seed(seed):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)  
#     torch.cuda.manual_seed(seed)  
#     torch.cuda.manual_seed_all(seed)  
#     os.environ['PYTHONHASHSEED'] = str(seed)
#     torch.backends.cudnn.enabled = True   
#     torch.backends.cudnn.deterministic = True  
#     torch.backends.cudnn.benchmark = False 
#     os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
#     print(f'CUBLAS_WORKSPACE_CONFIG: {os.environ["CUBLAS_WORKSPACE_CONFIG"]}') # You can print it out to see if the value has been set successfully
#     torch.use_deterministic_algorithms(True) 
    
# set_random_seed(42) # because obviously

In [4]:
dict_j = {1:'01', 2:'02', 3:'03', 4:'04', 5:'05', 6:'06', 7:'07', 8:'08', 9:'09', 10:'10'}

sage_aggr = hyperparameters['sage_aggr'] # 'sum'
sage_norm = hyperparameters['sage_norm'] # 'sum'
heteroconv_aggr = hyperparameters['heteroconv_aggr'] #'sum', 'cat'
heteroconv_aggr_1 = hyperparameters['heteroconv_aggr_1'] #'sum', 'cat'
heteroconv_aggr_2 = hyperparameters['heteroconv_aggr_2'] #'sum', 'cat'
heteroconv_aggr_3 = hyperparameters['heteroconv_aggr_3'] #'sum', 'cat'
node_mode = hyperparameters['node_mode'] # 'id' or 'degree'
sage_norm = hyperparameters['sage_norm']
Heads = hyperparameters['Heads']
ana_edges = hyperparameters['ana_edges']
string_threshold = hyperparameters['string_threshold']
# conv_number = hyperparameters['conv_number']
setting = hyperparameters['setting']
# conv_type = hyperparameters['conv_type']
heads = hyperparameters['Heads']
# activation = hyperparameters['activation']
global_loader_neighbors = hyperparameters['global_loader_neighbors']


t0 = time.time()

old_path = os.getcwd()
new_path = old_path + '/../../'
os.chdir(new_path)
current_path = os.getcwd()
data_raw_path = current_path + '/data/raw/'
data_processed_path = current_path + '/data/processed/'

In [6]:
# open the Yeasst_Knowledge_Graph
net = pd.read_csv(data_processed_path + f'yeast_knowledge_graph.tsv', sep = '\t', header = 0, dtype = {0: str, 1: str, 3: str, 4: str})
net = net.drop(net[(net['EdgeType'].isin(['PPI'])) & (net['EdgeScore'] < string_threshold)].index, axis = 0)
net = net.reset_index(drop = True)

all_genes = list(set().union(net[net['node1_type'] == 'GENE']['node1_ID'].unique(), net[net['node2_type'] == 'GENE']['node2_ID'].unique()))
# I could hardcode edges_names, edge_summary eveneutally
edges_names = net['EdgeType'].value_counts().to_dict()
edges_summary = []
for edge_name in edges_names.keys():
    tmp_list = [edge_name, 
                net[net['EdgeType'] == edge_name]['node1_type'].unique()[0], 
                net[net['EdgeType'] == edge_name]['node2_type'].unique()[0], 
                net[net['EdgeType'] == edge_name]['EdgeDirection'].unique()[0], 
                edges_names[edge_name]]
    # create a special case when there are edge_attributes
    # not currently using edges_attributes, but maybe in a future version 
    if sum(net[net['EdgeType'] == edge_name]['EdgeScore'].isna()) == 0:
        tmp_list.append(1)
    else:
        tmp_list.append(0)
    edges_summary.append(tmp_list) 

edges_types = list(set(net['EdgeType'].unique()))
nodes_types = list(set().union(net['node1_type'].unique(),net['node2_type'].unique()))
nodes_types.sort()
nodes_numbers = {}
for node_type in nodes_types:
    nodes_numbers[node_type] = len(set().union(net[net['node1_type'] == node_type]['node1_ID'].unique(), net[net['node2_type'] == node_type]['node2_ID'].unique()))

print(f"nodes_types: {nodes_types}")
print(f"nodes_numbers: {nodes_numbers}")

list_of_edges = net['EdgeType'].unique().tolist()

def Get_df_for_nodetype_id(df: pd.DataFrame, node_type : str) -> pd.DataFrame:
    node_type_list = list(set().union(df[df['node1_type'] == node_type]['node1_ID'].unique(), df[df['node2_type'] == node_type]['node2_ID'].unique()))
    node_type_list.sort()
    node_type_array = np.asarray(node_type_list)
    new_df = pd.DataFrame(data = {
        f"{node_type}_id": node_type_array,
        "mapped_id": pd.RangeIndex(len(node_type_array))
    })
    return(new_df)

nodes2id = {}
for type_of_node in nodes_types:
    nodes2id[type_of_node] = Get_df_for_nodetype_id(net, type_of_node)

edges_to_nodes = {}
for type_of_edge in edges_summary:
    edges_to_nodes[type_of_edge[0]] = [type_of_edge[1], type_of_edge[2]] 

def Get_df_for_edgetype(df: pd.DataFrame, type_of_edge: str, edge_score: bool = False) -> pd.DataFrame:
    node1_type = edges_to_nodes[type_of_edge][0]
    node2_type = edges_to_nodes[type_of_edge][1]
    new_df = df[df['EdgeType'] == type_of_edge].loc[:,["node1_ID", "node2_ID", "EdgeScore"]]
    new_df = pd.merge(new_df, nodes2id[node1_type], left_on = 'node1_ID', right_on = f"{node1_type}_id", how = 'left')
    new_df = new_df.rename(columns = {f"{node1_type}_id": f"{node1_type}1_id", 'mapped_id': 'node1_mapped_id'})
    new_df = pd.merge(new_df, nodes2id[node2_type], left_on = 'node2_ID', right_on = f"{node2_type}_id", how = 'left')
    new_df = new_df.rename(columns = {f"{node1_type}_id": f"{node1_type}2_id", 'mapped_id': 'node2_mapped_id'})
    if edge_score == False:
        new_df = new_df.drop('EdgeScore', axis = 1)
    return new_df

def Get_tensor_for_edges(df: pd.DataFrame, type_of_edge: str) -> torch.Tensor:
    node1_type = edges_to_nodes[type_of_edge][0]
    node2_type = edges_to_nodes[type_of_edge][1]
    new_df = df[df['EdgeType'] == type_of_edge]
    new_df = pd.merge(new_df, nodes2id[node1_type], left_on = 'node1_ID', right_on = f"{node1_type}_id", how = 'left')
    new_df = new_df.rename(columns = {"mapped_id": "node1_mapped_id"})
    new_df = pd.merge(new_df, nodes2id[node2_type], left_on = 'node2_ID', right_on = f"{node2_type}_id", how = 'left')
    new_df = new_df.rename(columns = {"mapped_id": "node2_mapped_id"})
    return torch.stack([torch.from_numpy(new_df['node1_mapped_id'].values), torch.from_numpy(new_df['node2_mapped_id'].values)], dim = 0)

def Get_tensor_and_rev_for_edges(df: pd.DataFrame, type_of_edge: str) -> torch.Tensor:
    node1_type = edges_to_nodes[type_of_edge][0]
    node2_type = edges_to_nodes[type_of_edge][1]
    new_df = df[df['EdgeType'] == type_of_edge]
    new_df = pd.merge(new_df, nodes2id[node1_type], left_on = 'node1_ID', right_on = f"{node1_type}_id", how = 'left')
    new_df = new_df.rename(columns = {"mapped_id": "node1_mapped_id"})
    new_df = pd.merge(new_df, nodes2id[node2_type], left_on = 'node2_ID', right_on = f"{node2_type}_id", how = 'left')
    new_df = new_df.rename(columns = {"mapped_id": "node2_mapped_id"})
    source, destination = torch.stack([torch.from_numpy(new_df['node1_mapped_id'].values), torch.from_numpy(new_df['node2_mapped_id'].values)], dim = 0)
    source_tot = torch.cat([source, destination])      
    destination_tot = torch.cat([destination, source])
    edge_index = torch.stack([source_tot, destination_tot], dim = 0)  
    return edge_index

edges2df = {}
for sublist in edges_summary:
    if sublist[5] == 1: 
        edges2df[sublist[0]] = Get_df_for_edgetype(net, sublist[0], True)
    else:
        edges2df[sublist[0]] = Get_df_for_edgetype(net, sublist[0], False)

def get_target_tensor():
    # for nodes, essential vs non_essential
    essential = pd.read_csv(data_processed_path + 'yeast_essential_nonessential_20240612.tsv', sep = '\t', header = 0)
    nodes2id['GENE'] = nodes2id['GENE'][['GENE_id', 'mapped_id']]
    nodes = nodes2id['GENE'].copy()
    nodes = nodes.merge(essential, how = 'left', left_on = 'GENE_id', right_on = 'gene_id')
    nodes.loc[nodes[nodes['type'] == 'essential'].index, 'target'] = 1
    nodes.loc[nodes[nodes['type'] == 'non_essential'].index, 'target'] = 0
    target_tensor = torch.from_numpy(nodes['target'].values)
    return target_tensor 

def add_strict_nodes(data: HeteroData) -> HeteroData:
    target_tensor = get_target_tensor()
    data['GENE'].num_nodes = len(nodes2id['GENE'])
    data['GENE'].y = target_tensor.type(torch.float32)
    for type_of_node in nodes_types:
        if type_of_node != 'GENE':
            data[type_of_node].num_nodes = len(nodes2id[type_of_node])
    return data

def create_masks_cross_val(data: HeteroData, split_seed: int = 1, cv_fold_to_test: int = 1) -> HeteroData:
    cv_folds = pd.read_csv(
        data_processed_path + 'data/processed/yeasts_n2n_6579_splits_seeds_cv.tsv',
        sep = '\t', header = 0)
    col_to_use = f"cv_seed_{split_seed}"
    folds = cv_folds[[col_to_use]]
    if cv_fold_to_test == 'leave_out_test':
        print("I will be training on the 10-folds and testing on the left-out test set.")
        train_folds = ['cv_1', 'cv_2', 'cv_3', 'cv_4', 'cv_5', 'cv_6', 'cv_7', 'cv_8', 'cv_9', 'cv_10']
        test_fold = 'leave_out_test'
    else:
        col_to_use = f"cv_seed_{split_seed}"
        folds = cv_folds[[col_to_use]]
        train_folds = ['cv_1', 'cv_2', 'cv_3', 'cv_4', 'cv_5', 'cv_6', 'cv_7', 'cv_8', 'cv_9', 'cv_10']
        test_fold = f'cv_{cv_fold_to_test}'
        train_folds.remove(test_fold)
        val_fold = 'leave_out_test'
    # get the indexes
    train_idx = folds[folds[col_to_use].isin(train_folds)].index
    test_idx = folds[folds[col_to_use] == test_fold].index
    if cv_fold_to_test == 'leave_out_test':
        pass
    else:
        val_idx = folds[folds[col_to_use].isin(val_fold)].index
    # initialize masks
    train_mask = torch.zeros(data['GENE'].x.shape[0])
    if cv_fold_to_test == 'leave_out_test':
        pass
    else:
        val_mask = torch.zeros(data['GENE'].x.shape[0])
    test_mask = torch.zeros(data['GENE'].x.shape[0])
    # create 1 at the specified index
    train_mask[train_idx] = 1
    if cv_fold_to_test == 'leave_out_test':
        pass
    else:
        val_mask[val_idx] = 1
    test_mask[test_idx] = 1
    # get the boolean tensor
    train_mask = train_mask > 0
    if cv_fold_to_test == 'leave_out_test':
        pass
    else:
        val_mask = val_mask > 0
    test_mask = test_mask > 0
    # create the masks to Data
    data['GENE'].train_mask = train_mask
    if cv_fold_to_test == 'leave_out_test':
        pass
    else:
        data['GENE'].val_mask = val_mask
    data['GENE'].test_mask = test_mask
    # return
    return data

# TODO update function to use 'net' as an input
def get_edges_of_type(data: HeteroData, type_of_edge: str, edge_attr: bool = False) -> HeteroData:
    node1_type = edges_to_nodes[type_of_edge][0]
    node2_type = edges_to_nodes[type_of_edge][1]
    new_df = net[net['EdgeType'] == type_of_edge]
    new_df = pd.merge(new_df, nodes2id[node1_type], left_on = 'node1_ID', right_on = f"{node1_type}_id", how = 'left')
    new_df = new_df.rename(columns = {"mapped_id": "node1_mapped_id"})
    new_df = pd.merge(new_df, nodes2id[node2_type], left_on = 'node2_ID', right_on = f"{node2_type}_id", how = 'left')
    new_df = new_df.rename(columns = {"mapped_id": "node2_mapped_id"})

## create a special case where I also take the edges attributes = edges scores
    if (edge_attr == True) and (type_of_edge in ['expressed_in', 'not_expressed_in', 'PPI', 'PPI_STRING', 'Neighbour', 'CoExp', 'DIDA']): 
        if type_of_edge in ['expressed_in', 'not_expressed_in']:
        # the edges 'expressed_in' and 'not_expressed_in' are directed because it's a link between different nodes types, I have to use T.ToUndirected() later
            edge_index = torch.stack([torch.from_numpy(new_df['node1_mapped_id'].values), torch.from_numpy(new_df['node2_mapped_id'].values)], dim = 0)
            #edge_attr = torch.from_numpy(edges2df[type_of_edge].EdgeScore.values).view(-1,1).type('torch.DoubleTensor')
            edge_attr = torch.from_numpy(edges2df[type_of_edge].EdgeScore.values).view(-1,1).type(torch.float32)
            data[node1_type, type_of_edge, node2_type].edge_index = edge_index
            data[node1_type, type_of_edge, node2_type].edge_attr = edge_attr#.view(-1)
        else:
        # the edges 'PPI', 'Neighbour', 'CoExp' and 'DIDA' are symetrical and between the same node type      
            source, destination = torch.stack([torch.from_numpy(new_df['node1_mapped_id'].values), torch.from_numpy(new_df['node2_mapped_id'].values)], dim = 0)
            source_tot = torch.cat([source, destination])      
            destination_tot = torch.cat([destination, source])  
            edge_index = torch.stack([source_tot, destination_tot], dim = 0)
            edge_attr = torch.from_numpy(edges2df[type_of_edge].EdgeScore.values).view(-1,1).type(torch.float32)
            edge_attr = torch.cat([edge_attr, edge_attr], dim = 0)
            data[node1_type, type_of_edge, node2_type].edge_index = edge_index
            data[node1_type, type_of_edge, node2_type].edge_attr = edge_attr#.view(-1)
## no edge_attr but directed edges, I have to use T.ToUndirected(merge = True) later for 'expressed_in' and 'not_expressed_in'
## use T.ToUndirected(merge = False) for 'ana_ana', 'biological_process', 'cellular_component', 'molecular_function', 'regulatory_proximal', 'regulatory_distal'
    elif (edge_attr == False) and (type_of_edge in ['expressed_in', 'not_expressed_in', 'gene_to_BP', 'gene_to_CC', 'gene_to_MF', 
                                                    'ana_ana', 'biological_process', 'cellular_component', 'molecular_function',
                                                    'regulatory_proximal', 'regulatory_distal'
                                                    ]):
        edge_index = torch.stack([torch.from_numpy(new_df['node1_mapped_id'].values), torch.from_numpy(new_df['node2_mapped_id'].values)], dim = 0)
        data[node1_type, type_of_edge, node2_type].edge_index = edge_index
## those edges are undirected
    elif (edge_attr == False) and (type_of_edge in ['PPI', 'PPI_STRING', 'paralog', 'Neighbour', 'CoExp', 'synthetic_lethal', 'synthetic_non_lethal', 'DIDA']):
        source, destination = torch.stack([torch.from_numpy(new_df['node1_mapped_id'].values), torch.from_numpy(new_df['node2_mapped_id'].values)], dim = 0)
        source_tot = torch.cat([source, destination])      
        destination_tot = torch.cat([destination, source])  
        edge_index = torch.stack([source_tot, destination_tot], dim = 0)
        data[node1_type, type_of_edge, node2_type].edge_index = edge_index
## if edge_attr is set to True for edges without an EdgeScore    
    else:
        raise Exception(f"the edge of type: {type_of_edge} doesn't have an edge_attribute. Run this function again with edge_attr set to False")
    
    return data

# in this section I will create dummy features for the nodes types that are not 'GENES' based on their node-degree, according to one edge-type
data = HeteroData()
data = add_strict_nodes(data)
data = get_edges_of_type(data, 'PPI')
data = get_edges_of_type(data, 'biological_process')
data = get_edges_of_type(data, 'cellular_component')
data = get_edges_of_type(data, 'molecular_function')
# those are dummy features for node_type == 'GENE', but I will not use it. TODO TOCLEAN
gg = Data()
gg.num_nodes = len(nodes2id['GENE'])
gg.edge_index = data.edge_index_dict[('GENE', 'PPI', 'GENE')]
df_gg = pd.DataFrame(gg.edge_index.t(), columns = ['source', 'dst'])
max_degree = max(df_gg.dst.value_counts().tolist()[0], df_gg.source.value_counts().tolist()[0])
gg = T.OneHotDegree(max_degree = max_degree)(gg)
gg.x_id = torch.eye(gg.num_nodes, dtype = torch.float32, requires_grad = False)

bp = Data()
bp.num_nodes = len(nodes2id['BP']) 
bp.edge_index = data.edge_index_dict[('BP', 'biological_process', 'BP')]
df_bp = pd.DataFrame(bp.edge_index.t(), columns = ['source', 'dst'])
max_degree = max(df_bp.dst.value_counts().tolist()[0], df_bp.source.value_counts().tolist()[0])
bp = T.OneHotDegree(max_degree = max_degree)(bp)
bp.x_id = torch.eye(bp.num_nodes, dtype = torch.float32, requires_grad = False)

cc = Data()
cc.num_nodes = len(nodes2id['CC']) 
cc.edge_index = data.edge_index_dict[('CC', 'cellular_component', 'CC')]
df_cc = pd.DataFrame(cc.edge_index.t(), columns = ['source', 'dst'])
max_degree = max(df_cc.dst.value_counts().tolist()[0], df_cc.source.value_counts().tolist()[0])
cc = T.OneHotDegree(max_degree = max_degree)(cc)
cc.x_id = torch.eye(cc.num_nodes, dtype = torch.float32, requires_grad = False)

mf = Data()
mf.num_nodes = len(nodes2id['MF']) 
mf.edge_index = data.edge_index_dict[('MF', 'molecular_function', 'MF')]
df_mf = pd.DataFrame(mf.edge_index.t(), columns = ['source', 'dst'])
max_degree = max(df_mf.dst.value_counts().tolist()[0], df_mf.source.value_counts().tolist()[0])
mf = T.OneHotDegree(max_degree = max_degree)(mf)
mf.x_id = torch.eye(mf.num_nodes, dtype = torch.float32, requires_grad = False)

def add_features(data: HeteroData, mode: str = 'degree', gene_mode: str = 'id') -> HeteroData:
    '''create features for the nodes types that are not 'GENE', either one-hot-encoding their node_degree or their id'''
    if mode == 'id':
        # data['ANATOMY'].x = ana.x_id
        data['BP'].x = bp.x_id
        data['CC'].x = cc.x_id
        data['MF'].x = mf.x_id        
    elif mode == 'degree':
        # data['ANATOMY'].x = ana.x
        data['BP'].x = bp.x
        data['CC'].x = cc.x
        data['MF'].x = mf.x
    else:
        raise Exception("Please select a mode 'id' or 'degree'")
    if gene_mode == 'id':
        data['GENE'].x = torch.eye(data['GENE'].num_nodes, dtype = torch.float32, requires_grad = False)
    elif gene_mode == 'degree':
        data['GENE'].x = gg.x
    
    return data

def create_bcmg_data(list_of_edges: list, use_edge_attr: bool = False, feature_mode: str = 'id', merge_choice: bool = True):
    edges_names = list_of_edges
    edges_with_attr = ['PPI', 'Neighbour', 'CoExp'] #
    # in data.edge_index_dict.keys(), an edge is a tuple (node1_type, relation, node2_type)
    edges_with_same_node_type = [
        ('GENE', 'PPI', 'GENE'),
        ('GENE', 'paralog', 'GENE'), 
        ('GENE', 'Neighbour', 'GENE'), 
        ('GENE', 'CoExp', 'GENE'), 
        ('BP', 'biological_process', 'BP'), 
        ('CC', 'cellular_component', 'CC'), 
        ('MF', 'molecular_function', 'MF')]
    target_tensor = get_target_tensor()
    bool_attr = use_edge_attr
    data = HeteroData()
    data = add_strict_nodes(data)
    data = add_features(data, mode = feature_mode, gene_mode = gene_mode) # 

    if use_edge_attr == False:
        for edge_type in list_of_edges:   
            data = get_edges_of_type(data, edge_type)
    else: # use_edge_attr == True
    # for the edges in list_of_edges with edge_attr    
        for edge_type in list(set(edges_with_attr).intersection(edges_names)):
            data = get_edges_of_type(data, edge_type, edge_attr = True)
    # for the edges in list_of_edges without edge_attr
        for edge_type in list(set(edges_names) - set(edges_with_attr)):
            data = get_edges_of_type(data, edge_type, edge_attr = False)

    # ne pas remove les self-loops quand il y a des différents nodes types !! 
    # double check the edges concerned
    for key in data.edge_index_dict.keys():
        if key in edges_with_same_node_type:
            if U.contains_self_loops(data.edge_index_dict[key]) == True:
                data[key].edge_index = U.remove_self_loops(data.edge_index_dict[key])[0]
    # if merge = False, will create the reverse edge_type
    data = T.ToUndirected(merge = merge_choice)(data)

    bp_list = ['gene_to_BP', 'biological_process']
    cc_list = ['gene_to_CC', 'cellular_component']
    mf_list = ['gene_to_MF', 'molecular_function']

    if any(edge in edges_names for edge in bp_list):
        pass
    else:
        del data['BP']
    if any(edge in edges_names for edge in cc_list):
        pass
    else:
        del data['CC']    
    if any(edge in edges_names for edge in mf_list):
        pass
    else:
        del data['MF']

    return edges_names, data

# clean edges but do not coalesce
def clean_data(data: HeteroData, edge_attr: bool = False) -> HeteroData:
    
    edges_with_attr = [
        ('GENE', 'PPI', 'GENE'), 
        ('GENE', 'Neighbour', 'GENE'), 
        ('GENE', 'CoExp', 'GENE')]
    edges_with_same_node_type = [
        ('GENE', 'PPI', 'GENE'),
        ('GENE', 'paralog', 'GENE'), 
        ('GENE', 'Neighbour', 'GENE'), 
        ('GENE', 'CoExp', 'GENE'), 
        ('BP', 'biological_process', 'BP'), 
        ('BP', 'rev_biological_process', 'BP'), 
        ('CC', 'cellular_component', 'CC'), 
        ('CC', 'rev_cellular_component', 'CC'), 
        ('MF', 'molecular_function', 'MF'),
        ('MF', 'rev_molecular_function', 'MF')]
    # in data.edge_index_dict.keys(), an edge is a tuple of 3 strings: ('node1_type', 'relation_type', 'node2_type')    
    if edge_attr == False:
        for edge in data.edge_index_dict.keys():
            data[edge].edge_index = data[edge].edge_index.contiguous()
        
            if edge in edges_with_same_node_type:
                if U.contains_self_loops(data.edge_index_dict[edge]) == True:
                    data[edge].edge_index = U.remove_self_loops(data.edge_index_dict[edge])[0]    
    
    elif edge_attr == True:
        for edge in data.edge_index_dict.keys():
            if edge not in edges_with_attr:
                data[edge].edge_index = data[edge].edge_index.contiguous()

                if edge in edges_with_same_node_type:
                    if U.contains_self_loops(data.edge_index_dict[edge]) == True:
                        data[edge].edge_index = U.remove_self_loops(data.edge_index_dict[edge])[0]

            elif edge in edges_with_attr:
                data[edge].edge_index = data[edge].edge_index.contiguous()

                if edge in edges_with_same_node_type:
                    if U.contains_self_loops(data.edge_index_dict[edge]) == True:
                        data[edge].edge_index = U.remove_self_loops(data.edge_index_dict[edge])[0]
    
    return data 

nodes_types: ['BP', 'CC', 'GENE', 'MF']
nodes_numbers: {'BP': 27995, 'CC': 4040, 'GENE': 6579, 'MF': 11297}


## define models 

## create data

In [8]:
# create the data now
gene_mode = hyperparameters['gene_mode']
node_mode = hyperparameters['node_mode'] # 'id' or 'degree'
edge_features = hyperparameters['edge_features'] # True or False

list_of_edges, data = create_bcmg_data([
    'gene_to_BP', 'biological_process', 
    'gene_to_CC', 'cellular_component', 
    'gene_to_MF', 'molecular_function'
    ], feature_mode = node_mode, use_edge_attr = edge_features, merge_choice = False) 
list_of_edges_2, data_2 = create_bcmg_data([
    'PPI', 'paralog' 
    ], use_edge_attr = edge_features, merge_choice = True) 
for key in data_2.edge_index_dict.keys():
    data[key].edge_index = data_2[key].edge_index
if edge_features == True:
    for key in data_2.edge_attr_dict.keys():
        data[key].edge_attr = data_2[key].edge_attr
else:
    pass
list_of_edges = list_of_edges + list_of_edges_2
del data_2
del list_of_edges_2
data = clean_data(data)

# TO DO coalesce at this step, the edges for individual genes 
for edge in data.edge_index_dict.keys():
    data[edge].edge_index = coalesce(data[edge].edge_index)

# create the gene features
feat_1000 = pd.read_csv(data_processed_path + 'yeast_features_1000_20240717.tsv', header = 0, sep = '\t')
yeast_features_to_use = [
    'dn_ds', 'chemical_compound_accumulation','chronological_lifespan', 'competitive_fitness',
    'desiccation_resistance', 'haploinsufficient', 'heat_sensitivity', 'metal_resistance', 
    'oxidative_stress_resistance', 'replicative_lifespan', 'resistance_to_chemicals', 'respiratory_growth',
    'stress_resistance', 'toxin_resistance', 'utilization_of_nitrogen_source', 'vacuolar_morphology', 'vegetative_growth']
data['GENE'].x = torch.from_numpy(feat_1000[yeast_features_to_use].astype('float32').values)
target_tensor = get_target_tensor()
data['GENE'].y = target_tensor 

print(f"The gene features have shape: {data['GENE'].x.shape}")

# TODO refaire mieux 
yeast_genes = pd.read_csv(data_processed_path + 'yeast_essential_nonessential_20240612.tsv', header = 0, sep = '\t')
yeast_genes['mapped_id'] = pd.RangeIndex(len(yeast_genes))
yeast_genes.loc[yeast_genes[yeast_genes['type'] == 'essential'].index, 'target'] = 1
yeast_genes.loc[yeast_genes[yeast_genes['type'] == 'non_essential'].index, 'target'] = 0
yeast_genes = yeast_genes.rename(columns = {'gene_id': 'GENE_id'})
fd = yeast_genes.copy()

data['GENE'].num_nodes = len(data['GENE'].x)

for edge in data.edge_types:
    data[edge].edge_index = U.sort_edge_index(data[edge].edge_index)
data = data.to('cpu')

The gene features have shape: torch.Size([6579, 17])


In [9]:
print("Creating the new_nodes representing pairs of genes.")
yeast_genes = pd.read_csv(data_processed_path + 'yeast_essential_nonessential_20240612.tsv', header = 0, sep = '\t')

Creating the new_nodes representing pairs of genes.


In [10]:
yeast_genes

,gene_id,type
0,YAL001C,essential
1,YAL002W,non_essential
2,YAL003W,essential
3,YAL004W,non_essential
4,YAL005C,non_essential
...,...,...
6574,YPR201W,non_essential
6575,YPR202W,non_essential
6576,YPR203W,non_essential
6577,YPR204C-A,non_essential


In [11]:
nodes = nodes2id['GENE'].merge(yeast_genes, how='left', left_on='GENE_id', right_on='gene_id')
nodes = nodes.drop('gene_id', axis=1)

# open pairs_df

In [ ]:
def get_nop(df:pd.DataFrame, left_col: str, right_col:str):
    df['nodes_are_ordered'] = df[left_col] < df[right_col]
    ordered_index = df[df['nodes_are_ordered'] == True].index
    reverse_ordered_index = df[df['nodes_are_ordered'] == False].index
    df.loc[ordered_index, 'name_of_pair'] = df[left_col].astype(str) + '_' + df[right_col].astype(str)
    df.loc[reverse_ordered_index, 'name_of_pair'] = df[right_col].astype(str) + '_' + df[left_col].astype(str)
    return df

def split_name(name: str, side: str = 0):
    if '_' in name:
        new_name = name.split('_')[side]
    else:
        new_name = name
    return new_name 

In [25]:
# generate pairs

use_all_4262_genes = True

# use_all_4262_genes = True
print("Opening the interaction dataframe.")
genetic_interaction = pd.read_csv(data_processed_path + 'yeast_pairs_20240617.tsv', header = 0, sep = '\t')
print("Filtering it.")
# 15,662,490 pairs
pairs = genetic_interaction[genetic_interaction['file']=='NxN'] # 11,199,856 pairs
pairs = pairs.merge(nodes, how = 'left', left_on = 'node1_ID', right_on = 'GENE_id', suffixes = (None, '_node_1'))
pairs = pairs.drop('GENE_id', axis = 1)
pairs = pairs.rename(columns={'mapped_id': 'mapped_id_left', 'type': 'type_node_1'})
pairs = pairs.merge(nodes, how = 'left', left_on = 'node2_ID', right_on = 'GENE_id', suffixes = (None, '_node_2'))
pairs = pairs.drop('GENE_id', axis = 1)
pairs = pairs.rename(columns={'mapped_id': 'mapped_id_right', 'type': 'type_node_2'})
# 11,199,856
pairs = pairs.drop(pairs[(pairs['type_node_1'] == 'essential') | (pairs['type_node_2'] == 'essential')].index, axis = 0) # drop 730,988  there are 10,468,868 remaining
pairs = get_nop(pairs, left_col = 'node1_ID', right_col = 'node2_ID')
pairs = pairs.sort_values(by=['name_of_pair', 'fitness_12'], ascending=False)
print('I am removing ALL duplicated pairs.')
pairs = pairs.drop_duplicates(subset = 'name_of_pair', keep = False, ignore_index = True) # ou keep = False ? 'first' ?drop them all ? 
print("Filtering finished.")
# 4,809,647 remaining 

# if use_all_4262_genes == True:
#     pass
# else: # elif use_all_4262_genes == False
#     print('I will remove some SL and SNL pairs in order to have more unused non_essential genes that will be used in the n2p approach.')
#     print(f"I will use the DMF threshold of 0.19 to define a small number of SL pairs composed of fewer genes than the total.")
#     sl_019 = pairs[pairs['fitness_12']<=0.19]
#     sl_genes_019 = list(set(sl_019['node1_ID'].unique().tolist()+sl_019['node2_ID'].unique().tolist()))
#     sl_genes_019.sort()
#     print(f"There are {len(sl_genes_019)} individual genes in the SL pairs with DMF <= 0.19")
#     nodes.loc[:,'in_pairs_SL_0.19'] = 0
#     nodes.loc[nodes[nodes['GENE_id'].isin(sl_genes_019)].index, 'in_pairs_SL_0.19'] = 1
#     available_non_essential_genes = len(nodes[(nodes['type']=='non_essential')&(nodes['in_pairs_SL_0.19']==0)])
#     print(f"There are {available_non_essential_genes} non_essential genes that are unused in pairs.")
#     # nodes[nodes['in_pairs_SL_0.19']==0]['type'].value_counts()
#     n2p_train = nodes[nodes['in_pairs_SL_0.19']==0]['GENE_id'].tolist()
#     old_length = len(pairs)
#     pairs = pairs.drop(pairs[(pairs['node1_ID'].isin(n2p_train))|(pairs['node2_ID'].isin(n2p_train))].index, axis = 0)
#     pairs = pairs.reset_index(drop=True)
#     new_length = len(pairs)
#     print(f"I deleted {old_length-new_length} pairs in total.")

Opening the interaction dataframe.
Filtering it.
I am removing ALL duplicated pairs.
Filtering finished.


In [22]:
pairs

,gene_1,gene_2,interaction,p_val,fitness_1,fitness_2,fitness_12,fitness_12_sd,node1_ID,node2_ID,file,mapped_id_left,type_node_1,mapped_id_right,type_node_2,nodes_are_ordered,name_of_pair
0,YPR184W,YPR122W,-0.0237,0.42010,1.0162,0.9850,0.9773,0.1135,YPR184W,YPR122W,NxN,6557,non_essential,6487,non_essential,False,YPR122W_YPR184W
1,YPR184W,YPR121W,-0.0260,0.33870,1.0162,0.9812,0.9711,0.0543,YPR184W,YPR121W,NxN,6557,non_essential,6486,non_essential,False,YPR121W_YPR184W
2,YPR119W,YPR201W,-0.0097,0.33780,1.0220,0.9880,1.0000,0.0156,YPR119W,YPR201W,NxN,6484,non_essential,6574,non_essential,True,YPR119W_YPR201W
3,YPR119W,YPR200C,-0.0110,0.35070,1.0220,1.0345,1.0463,0.0198,YPR119W,YPR200C,NxN,6484,non_essential,6573,non_essential,True,YPR119W_YPR200C
4,YPR184W,YPR119W,-0.0231,0.31860,1.0162,1.0220,1.0154,0.0385,YPR184W,YPR119W,NxN,6557,non_essential,6484,non_essential,False,YPR119W_YPR184W
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4809642,YAL002W,YBL008W,0.0532,0.07149,0.7552,0.9370,0.7608,0.0241,YAL002W,YBL008W,NxN,1,non_essential,125,non_essential,True,YAL002W_YBL008W
4809643,YBL006C,YAL002W,-0.0544,0.09995,0.7765,0.7552,0.5320,0.0336,YBL006C,YAL002W,NxN,122,non_essential,1,non_essential,False,YAL002W_YBL006C
4809644,YAL002W,YBL005W,-0.0228,0.29790,0.7552,1.0248,0.7512,0.0343,YAL002W,YBL005W,NxN,1,non_essential,121,non_essential,True,YAL002W_YBL005W
4809645,YAL063C-A,YAL002W,-0.0589,0.11330,0.9785,0.7552,0.6801,0.0372,YAL063C-A,YAL002W,NxN,73,non_essential,1,non_essential,False,YAL002W_YAL063C-A


In [36]:
import math

def sample_large_SNL_pairs_with_split_50_and_80(nodes:pd.DataFrame, pairs: pd.DataFrame, DMF_t: float = 0.4):
    for col_name in ['mask_05', 'mask_08']:
        if col_name in pairs.columns:
            pairs = pairs.drop(col_name, axis=1)
    if 'name_of_pair' not in pairs.columns:
        pairs = pairs.rename(columns={'GENE_id':'name_of_pair'})
    # Generating the SNL pairs
    print("I will not use any interaction value to define SNL pairs.")
    # snl = pairs[(pairs['fitness_12']>0.98)&(pairs['fitness_12']<1.02)&(pairs['interaction']>=-0.1)&(pairs['interaction']<=0.1)]
    # snl = pairs[(pairs['fitness_12']>0.999)&(pairs['fitness_12']<1.001)]
    snl = pairs[(pairs['fitness_12']>0.98)&(pairs['fitness_12']<1.02)]
    snl = snl.reset_index(drop=True)
    snl.loc[:,['type']] = 'SNL'
    sl = pairs[pairs['fitness_12']<= DMF_t]
    sl = sl.reset_index(drop=True)
    sl.loc[:,['type']] = 'SL'
    sl_genes = list(set(sl['node1_ID'].unique().tolist() + sl['node2_ID'].unique().tolist()))
    sl_genes.sort()
    print(f"There are {len(sl)} SL pairs, composed of {len(sl_genes)} individual non_essential genes.")

    ## with split 80/20
    print("XXXX"*20,"\nFor the 80/20 split:")
    genes_ratio_in_train = 0.8
    print(f"I will use {100*genes_ratio_in_train}% of the {len(sl_genes)} genes to be used exclusively in the train set.")
    random.seed(42)
    genes_train_80_percent = random.sample(sl_genes, k = math.ceil(genes_ratio_in_train*len(sl_genes)))
    genes_train_80_percent.sort()
    print(f"I will use {len(genes_train_80_percent)} genes in the train set and {len(sl_genes)-len(genes_train_80_percent)} genes in the test set.")
    # nodes.loc[nodes[(~nodes['GENE_id'].isin(sl_genes_train_80_percent))&(nodes['GENE_id'].isin(sl_genes))].index, 'in_pairs_08'] = 'SL_test'
    sl.loc[:,['mask_08']] = 'val'
    sl.loc[sl[(sl['node1_ID'].isin(genes_train_80_percent))&(sl['node2_ID'].isin(genes_train_80_percent))].index, 'mask_08'] = 'train'
    sl.loc[sl[(~sl['node1_ID'].isin(genes_train_80_percent))&(~sl['node2_ID'].isin(genes_train_80_percent))].index, 'mask_08'] = 'test'
    train_pairs_08 = len(sl[sl['mask_08']=='train'])
    val_pairs_08 = len(sl[sl['mask_08']=='val'])
    test_pairs_08 = len(sl[sl['mask_08']=='test'])
    # print(sl['mask'].value_counts())
    print(f"With the random sampling of individual genes that I have done, there are:\n",
    f"- {train_pairs_08} SL pairs in the train_set\n",
    f"- {test_pairs_08} SL pairs in the test set\n",
    f"- {val_pairs_08} SL pairs with one gene in the train set and the other one in the test set.")

    ## with split 50/50
    print("XXXX"*20,"\nFor the 50/50 split:")
    genes_ratio_in_train = 0.5
    print(f"I will use {100*genes_ratio_in_train}% of the {len(sl_genes)} genes to be used exclusively in the train set.")
    # the genes in the in the 50/50 split will be sampled from the 80/20 split set  
    random.seed(42)
    genes_train_50_percent = random.sample(genes_train_80_percent, k = math.ceil(genes_ratio_in_train*len(sl_genes)))
    # sl_genes_train_50_percent = random.sample(sl_genes, k = int(sl_genes_ratio_in_train*len(sl_genes)))
    genes_train_50_percent.sort()
    print(f"I will use {len(genes_train_50_percent)} genes in the train set and {len(sl_genes)-len(genes_train_50_percent)} genes in the test set.")
    # nodes.loc[nodes[(~nodes['GENE_id'].isin(sl_genes_train_50_percent))&(nodes['GENE_id'].isin(sl_genes))].index, 'in_pairs_05'] = 'in_test_SL_pairs'
    sl.loc[:,['mask_05']] = 'val'
    sl.loc[sl[(sl['node1_ID'].isin(genes_train_50_percent))&(sl['node2_ID'].isin(genes_train_50_percent))].index, 'mask_05'] = 'train'
    sl.loc[sl[(~sl['node1_ID'].isin(genes_train_50_percent))&(~sl['node2_ID'].isin(genes_train_50_percent))].index, 'mask_05'] = 'test'
    train_pairs_05 = len(sl[sl['mask_05']=='train'])
    val_pairs_05 = len(sl[sl['mask_05']=='val'])
    test_pairs_05 = len(sl[sl['mask_05']=='test'])
    # print(sl['mask'].value_counts())
    print(f"With the random sampling of individual genes that I have done, there are:\n",
    f"- {train_pairs_05} SL pairs in the train_set\n",
    f"- {test_pairs_05} SL pairs in the test set\n",
    f"- {val_pairs_05} SL pairs with one gene in the train set and the other one in the test set.")
    print("XXXX"*20)
    ##############
    genes_train_unique_in_80 = list(set(genes_train_80_percent)-set(genes_train_50_percent))
    genes_train_unique_in_80.sort()
    print(f"There are {len(genes_train_unique_in_80)} genes that are present in the train_set in the 80/20 split but in the test set for the 50/50 split.")
    print(f"The sampling of SNL pairs for the train_set for the 80/20 split will respect the relative distribution of genes.")

    print("XXXX"*20)
    ##############
    nodes.loc[:,['in_pairs_08']] = 'not_in_pairs'
    nodes.loc[:,['in_pairs_05']] = 'not_in_pairs'
    genes_test_20_percent = list(set(sl_genes)-set(genes_train_80_percent))
    nodes.loc[nodes[nodes['GENE_id'].isin(genes_train_80_percent)].index, ['in_pairs_08']] = 'train_pairs'
    nodes.loc[nodes[nodes['GENE_id'].isin(genes_test_20_percent)].index, ['in_pairs_08']] = 'test_pairs'
    genes_test_50_percent = list(set(sl_genes)-set(genes_train_50_percent))
    nodes.loc[nodes[nodes['GENE_id'].isin(genes_train_50_percent)].index, ['in_pairs_05']] = 'train_pairs'
    nodes.loc[nodes[nodes['GENE_id'].isin(genes_test_50_percent)].index, ['in_pairs_05']] = 'test_pairs'

    assert train_pairs_05 < train_pairs_08
    print(f"There will be {train_pairs_05} SNL pairs that will always be in the train set.")
    print(f"There will be {train_pairs_08-train_pairs_05} SNL pairs that will be in the train/val/test sets depending on the masks.")
    print(f"There will be {test_pairs_08} pairs that will always be in the test set.")
    print("XXXX"*20)

    # train_pairs_05 = 37488
    print(f" There are {train_pairs_05} SNL pairs that are in the train set for both the 80/20 and the 50/50 splits.")
    snl_always_train_index = list(snl[(snl['node1_ID'].isin(genes_train_50_percent))&(snl['node2_ID'].isin(genes_train_50_percent))].index)
    random.seed(42)
    snl_always_train_sampled = random.sample(snl_always_train_index, k = train_pairs_05)
    snl.loc[snl_always_train_sampled,['mask_08']] = 'train'
    snl.loc[snl_always_train_sampled,['mask_05']] = 'train'

    # test_pairs_08 = 6455
    print(f" There are {test_pairs_08} SNL pairs that are in the test set for both the 80/20 and the 50/50 splits.")
    snl_always_test_index = list(snl[(~snl['node1_ID'].isin(genes_train_80_percent))&(~snl['node2_ID'].isin(genes_train_80_percent))].index)
    random.seed(42)
    snl_always_test_sampled = random.sample(snl_always_test_index, k = test_pairs_08)
    snl.loc[snl_always_test_sampled,['mask_08']] = 'test'
    snl.loc[snl_always_test_sampled,['mask_05']] = 'test'

    missing_train_08_pairs = train_pairs_08-train_pairs_05
    train_08_val_05 = math.ceil(train_pairs_08*2*5/8-train_pairs_05*2)
    train_08_test_05 = missing_train_08_pairs - train_08_val_05

    # sample the pairs that will be in the train_set for mask_08 and val_set for mask_05
    print(f" There are {train_pairs_08} SNL pairs in the train set for the 80/20 split.\n",
        f"And I have already sampled {train_pairs_05} of them.\n",
        f"So there remain {missing_train_08_pairs} pairs to be sampled.\n",
        f"I want 5/8 so {5/8*100:0.3f}% of the {train_pairs_08} x 2 = {train_pairs_08*2} genes total, so {int(train_pairs_08*2*5/8)} genes to come from the set of {len(genes_train_50_percent)} genes used as train set for the 50/50 split.\n",
        f"I have already sampled {train_pairs_05} pairs where the 2 genes come from this small set of {len(genes_train_50_percent)} so there are already {train_pairs_05*2} genes out of the {int(train_pairs_08*2*5/8)}.\n",
        f"So I am missing {int(train_08_val_05)} genes from this small set.\n",
        f"So I will sample {int(train_08_val_05)} pairs where 1 gene come from this small set of {len(genes_train_50_percent)} and the other one comes from the set of {len(genes_train_unique_in_80)} genes.\n",
        f"These {int(train_08_val_05)} pairs will be in the val_set for the mask_05.\n",
        f"Finally I need to sample {train_08_test_05} pairs where both genes come from the set of {len(genes_train_unique_in_80)} genes.\n",
        f"And they will be in the test_set for the 50/50 split.")

    print(f"There are {int(train_08_val_05)} SNL pairs that are in the train set for the 80/20 split and in the val set for the 50/50 split.")
    snl_train_val_index = list(snl[
        ((snl['node1_ID'].isin(genes_train_50_percent))&(snl['node2_ID'].isin(genes_train_unique_in_80)))
        |
        ((snl['node1_ID'].isin(genes_train_unique_in_80))&(snl['node2_ID'].isin(genes_train_50_percent)))
        ].index)
    random.seed(42)
    snl_train_val_sampled = random.sample(snl_train_val_index, k = int(train_08_val_05))
    snl.loc[snl_train_val_sampled,['mask_08']] = 'train'
    snl.loc[snl_train_val_sampled,['mask_05']] = 'val'

    print(f"There are {int(train_08_test_05)} SNL pairs that are in the train set for the 80/20 split and in the test set for the 50/50 split.")
    snl_train_test_index = list(snl[(snl['node1_ID'].isin(genes_train_unique_in_80))&(snl['node2_ID'].isin(genes_train_unique_in_80))].index)
    random.seed(42)
    snl_train_test_sampled = random.sample(snl_train_test_index, k = int(train_08_test_05))
    snl.loc[snl_train_test_sampled,['mask_08']] = 'train'
    snl.loc[snl_train_test_sampled,['mask_05']] = 'test'
    
    # TODO change this val one
    # do the missing test_pairs_05 first ?
    
    missing_val_pairs_05 = val_pairs_05 - len(snl[snl['mask_05']=='val'])
    # 20918 pairs in val 05 and val 08
    # 1 gene in genes_50_percent and 1 gene not in genes_80_percent = test_set (20_percent)
    print(f"There are {int(missing_val_pairs_05)} SNL pairs that are in the val set for both the 80/20 and the 50/50 splits.")
    snl_always_val_index = list(snl[
        ((snl['node1_ID'].isin(genes_train_50_percent))&(~snl['node2_ID'].isin(genes_train_80_percent)))
        |
        ((~snl['node1_ID'].isin(genes_train_80_percent))&(snl['node2_ID'].isin(genes_train_50_percent)))
        ].index)
    random.seed(42)
    snl_always_val_sampled = random.sample(snl_always_val_index, k = int(missing_val_pairs_05))
    snl.loc[snl_always_val_sampled,['mask_08']] = 'val'
    snl.loc[snl_always_val_sampled,['mask_05']] = 'val'

    missing_val_pairs_08 = val_pairs_08 - len(snl[snl['mask_08']=='val'])
    missing_test_pairs_05 = test_pairs_05 - len(snl[snl['mask_05']=='test'])
    assert missing_val_pairs_08 == missing_test_pairs_05
    ## 33182 pairs in val_08 and test_05
    # 1 gene in genes_unique_in_80 and 1 gene not in genes_train_80
    print(f"There are {int(missing_val_pairs_08)} SNL pairs that are in the val set for the 80/20 split and in the test set for the 50/50 split.")
    snl_val_test_index = list(snl[
        ((snl['node1_ID'].isin(genes_train_unique_in_80))&(~snl['node2_ID'].isin(genes_train_80_percent)))
        |
        ((~snl['node1_ID'].isin(genes_train_80_percent))&(snl['node2_ID'].isin(genes_train_unique_in_80)))
    ].index)
    random.seed(42)
    snl_val_test_sampled = random.sample(snl_val_test_index, k = int(missing_val_pairs_08))
    snl.loc[snl_val_test_sampled,['mask_08']] = 'val'
    snl.loc[snl_val_test_sampled,['mask_05']] = 'test'

    nodes = nodes[['GENE_id', 'mapped_id', 'type', 'in_pairs_05', 'in_pairs_08']]
    nodes.loc[:, ['mask_05']] = 'individual_genes'
    nodes.loc[:, ['mask_08']] = 'individual_genes'

    # sl = sl[~sl['mask_05'].isna()]
    # sl = sl.reset_index(drop=True)
    snl = snl[~snl['mask_05'].isna()]
    snl = snl.reset_index(drop=True)
    sl = sl[['name_of_pair', 'type', 'mask_05', 'mask_08', 'fitness_12', 'interaction']]
    snl = snl[['name_of_pair', 'type', 'mask_05', 'mask_08', 'fitness_12', 'interaction']]
    pairs = pd.concat((sl, snl), axis=0)
    pairs = pairs.rename(columns={'name_of_pair': 'GENE_id'})
    pairs = pairs.sort_values(by='GENE_id', ascending=True, ignore_index=True)

    new_nodes = pd.concat([nodes, pairs], axis = 0)
    new_nodes = new_nodes.reset_index(drop = True)
    new_nodes['mapped_id'] = pd.RangeIndex(len(new_nodes))

    new_nodes.loc[new_nodes[new_nodes['type'] == 'essential'].index, 'target'] = 1
    new_nodes.loc[new_nodes[new_nodes['type'] == 'non_essential'].index, 'target'] = 0
    new_nodes.loc[new_nodes[new_nodes['type'] == 'SL'].index, 'target'] = 1
    new_nodes.loc[new_nodes[new_nodes['type'] == 'SNL'].index, 'target'] = 0
    fd = new_nodes.copy()

    # map() series https://stackoverflow.com/questions/34962104/how-can-i-use-the-apply-function-for-a-single-column 
    fd['left_side'] = fd['GENE_id'].map(lambda x: split_name(x, side = 0))
    fd['right_side'] = fd['GENE_id'].map(lambda x: split_name(x, side = 1))
    fd = fd.merge(nodes[['GENE_id', 'mapped_id']], how = 'left', left_on = 'left_side', right_on = 'GENE_id', suffixes = (None, "_left"))
    fd = fd.merge(nodes[['GENE_id', 'mapped_id']], how = 'left', left_on = 'right_side', right_on = 'GENE_id', suffixes = (None, "_right"))
    fd = fd.drop(['GENE_id_left', 'GENE_id_right'], axis = 1)

    return nodes, fd

## 4261 genes

### DMF_t = 0.5

In [37]:
nodes = nodes[['GENE_id', 'mapped_id', 'type']]

In [38]:
nodes, fd = sample_large_SNL_pairs_with_split_50_and_80(nodes=nodes, pairs=pairs, DMF_t=0.5)

I will not use any interaction value to define SNL pairs.
There are 290352 SL pairs, composed of 4261 individual non_essential genes.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 80/20 split:
I will use 80.0% of the 4261 genes to be used exclusively in the train set.
I will use 3409 genes in the train set and 852 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 200220 SL pairs in the train_set
 - 8048 SL pairs in the test set
 - 82084 SL pairs with one gene in the train set and the other one in the test set.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 50/50 split:
I will use 50.0% of the 4261 genes to be used exclusively in the train set.
I will use 2131 genes in the train set and 2130 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 81379 SL pairs in the train_set
 - 63680 SL pairs in the test s

In [39]:
print(fd[fd['type']=='SL']['mask_05'].value_counts())
print(fd[fd['type']=='SNL']['mask_05'].value_counts())
print(fd[fd['type']=='SL']['mask_08'].value_counts())
print(fd[fd['type']=='SNL']['mask_08'].value_counts())

mask_05
val      145293
train     81379
test      63680
Name: count, dtype: int64
mask_05
val      145293
train     81379
test      63680
Name: count, dtype: int64
mask_08
train    200220
val       82084
test       8048
Name: count, dtype: int64
mask_08
train    200220
val       82084
test       8048
Name: count, dtype: int64


In [40]:
fd.to_csv(
    path_or_buf=data_processed_path + '20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.5_4261_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=True, index=False, sep='\t', mode='x')

### DMF_t = 0.4

In [3]:
fd = pd.read_csv(
    data_processed_path + '20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.5_4261_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=0, sep='\t', dtype={3:str, 4:str}
)
pairs = fd[fd['GENE_id'].str.contains('_')]
nodes = nodes[['GENE_id', 'mapped_id', 'type']]
pairs = pairs.rename(columns={'mapped_id_left':'node1_ID', 'mapped_id_right':'node2_ID'})

pairs['type'].value_counts()

NameError: name 'nodes' is not defined

In [47]:
nodes, fd = sample_large_SNL_pairs_with_split_50_and_80(nodes=nodes, pairs=pairs, DMF_t=0.4)

I will not use any interaction value to define SNL pairs.
There are 164260 SL pairs, composed of 4261 individual non_essential genes.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 80/20 split:
I will use 80.0% of the 4261 genes to be used exclusively in the train set.
I will use 3409 genes in the train set and 852 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 118779 SL pairs in the train_set
 - 3180 SL pairs in the test set
 - 42301 SL pairs with one gene in the train set and the other one in the test set.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 50/50 split:
I will use 50.0% of the 4261 genes to be used exclusively in the train set.
I will use 2131 genes in the train set and 2130 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 50065 SL pairs in the train_set
 - 31966 SL pairs in the test s

In [48]:
print(fd[fd['type']=='SL']['mask_05'].value_counts())
print(fd[fd['type']=='SNL']['mask_05'].value_counts())
print(fd[fd['type']=='SL']['mask_08'].value_counts())
print(fd[fd['type']=='SNL']['mask_08'].value_counts())

mask_05
val      82229
train    50065
test     31966
Name: count, dtype: int64
mask_05
val      82229
train    50065
test     31966
Name: count, dtype: int64
mask_08
train    118779
val       42301
test       3180
Name: count, dtype: int64
mask_08
train    118779
val       42301
test       3180
Name: count, dtype: int64


In [49]:
fd.to_csv(
    path_or_buf=data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.4_4261_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=True, index=False, sep='\t', mode='x')

### DMF_t = 0.3

In [50]:
fd = pd.read_csv(
    data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.4_4261_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=0, sep='\t', dtype={3:str, 4:str}
)
pairs = fd[fd['GENE_id'].str.contains('_')]
nodes = nodes[['GENE_id', 'mapped_id', 'type']]
pairs = pairs.rename(columns={'mapped_id_left':'node1_ID', 'mapped_id_right':'node2_ID'})

pairs['type'].value_counts()

type
SL     164260
SNL    164260
Name: count, dtype: int64

In [51]:
nodes, fd = sample_large_SNL_pairs_with_split_50_and_80(nodes=nodes, pairs=pairs, DMF_t=0.3)

I will not use any interaction value to define SNL pairs.
There are 72718 SL pairs, composed of 4254 individual non_essential genes.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 80/20 split:
I will use 80.0% of the 4254 genes to be used exclusively in the train set.
I will use 3404 genes in the train set and 850 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 49815 SL pairs in the train_set
 - 2122 SL pairs in the test set
 - 20781 SL pairs with one gene in the train set and the other one in the test set.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 50/50 split:
I will use 50.0% of the 4254 genes to be used exclusively in the train set.
I will use 2127 genes in the train set and 2127 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 15988 SL pairs in the train_set
 - 20345 SL pairs in the test set

In [52]:
print(fd[fd['type']=='SL']['mask_05'].value_counts())
print(fd[fd['type']=='SNL']['mask_05'].value_counts())
print(fd[fd['type']=='SL']['mask_08'].value_counts())
print(fd[fd['type']=='SNL']['mask_08'].value_counts())

mask_05
val      36385
test     20345
train    15988
Name: count, dtype: int64
mask_05
val      36385
test     20345
train    15988
Name: count, dtype: int64
mask_08
train    49815
val      20781
test      2122
Name: count, dtype: int64
mask_08
train    49815
val      20781
test      2122
Name: count, dtype: int64


In [53]:
fd.to_csv(
    path_or_buf=data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.3_4261_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=True, index=False, sep='\t', mode='x')

### DMF_t = 0.2

In [54]:
fd = pd.read_csv(
    data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.3_4261_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=0, sep='\t', dtype={3:str, 4:str}
)
pairs = fd[fd['GENE_id'].str.contains('_')]
nodes = nodes[['GENE_id', 'mapped_id', 'type']]
pairs = pairs.rename(columns={'mapped_id_left':'node1_ID', 'mapped_id_right':'node2_ID'})

pairs['type'].value_counts()

type
SL     72718
SNL    72718
Name: count, dtype: int64

In [ ]:
nodes, fd = sample_large_SNL_pairs_with_split_50_and_80(nodes=nodes, pairs=pairs, DMF_t=0.2)

## 3841 genes

In [56]:
# generate pairs

use_all_4262_genes = False

# use_all_4262_genes = True
print("Opening the interaction dataframe.")
genetic_interaction = pd.read_csv(data_processed_path+'yeast_pairs_20240617.tsv', header = 0, sep = '\t')
print("Filtering it.")
# 15,662,490 pairs
pairs = genetic_interaction[genetic_interaction['file']=='NxN'] # 11,199,856 pairs
pairs = pairs.merge(nodes, how = 'left', left_on = 'node1_ID', right_on = 'GENE_id', suffixes = (None, '_node_1'))
pairs = pairs.drop('GENE_id', axis = 1)
pairs = pairs.rename(columns={'mapped_id': 'mapped_id_left', 'type': 'type_node_1'})
pairs = pairs.merge(nodes, how = 'left', left_on = 'node2_ID', right_on = 'GENE_id', suffixes = (None, '_node_2'))
pairs = pairs.drop('GENE_id', axis = 1)
pairs = pairs.rename(columns={'mapped_id': 'mapped_id_right', 'type': 'type_node_2'})
# 11,199,856
pairs = pairs.drop(pairs[(pairs['type_node_1'] == 'essential') | (pairs['type_node_2'] == 'essential')].index, axis = 0) # drop 730,988  there are 10,468,868 remaining
pairs = get_nop(pairs, left_col = 'node1_ID', right_col = 'node2_ID')
pairs = pairs.sort_values(by=['name_of_pair', 'fitness_12'], ascending=False)
print('I am removing ALL duplicated pairs.')
pairs = pairs.drop_duplicates(subset = 'name_of_pair', keep = False, ignore_index = True) # ou keep = False ? 'first' ?drop them all ? 
print("Filtering finished.")
# 4,809,647 remaining 

if use_all_4262_genes == True:
    pass
else: # elif use_all_4262_genes == False
    print('I will remove some SL and SNL pairs in order to have more unused non_essential genes that will be used in the n2p approach.')
    print(f"I will use the DMF threshold of 0.19 to define a small number of SL pairs composed of fewer genes than the total.")
    sl_019 = pairs[pairs['fitness_12']<=0.19]
    sl_genes_019 = list(set(sl_019['node1_ID'].unique().tolist()+sl_019['node2_ID'].unique().tolist()))
    sl_genes_019.sort()
    print(f"There are {len(sl_genes_019)} individual genes in the SL pairs with DMF <= 0.19")
    nodes.loc[:,'in_pairs_SL_0.19'] = 0
    nodes.loc[nodes[nodes['GENE_id'].isin(sl_genes_019)].index, 'in_pairs_SL_0.19'] = 1
    available_non_essential_genes = len(nodes[(nodes['type']=='non_essential')&(nodes['in_pairs_SL_0.19']==0)])
    print(f"There are {available_non_essential_genes} non_essential genes that are unused in pairs.")
    # nodes[nodes['in_pairs_SL_0.19']==0]['type'].value_counts()
    n2p_train = nodes[nodes['in_pairs_SL_0.19']==0]['GENE_id'].tolist()
    old_length = len(pairs)
    pairs = pairs.drop(pairs[(pairs['node1_ID'].isin(n2p_train))|(pairs['node2_ID'].isin(n2p_train))].index, axis = 0)
    pairs = pairs.reset_index(drop=True)
    new_length = len(pairs)
    print(f"I deleted {old_length-new_length} pairs in total.")

Opening the interaction dataframe.
Filtering it.
I am removing ALL duplicated pairs.
Filtering finished.
I will remove some SL and SNL pairs in order to have more unused non_essential genes that will be used in the n2p approach.
I will use the DMF threshold of 0.19 to define a small number of SL pairs composed of fewer genes than the total.
There are 3841 individual genes in the SL pairs with DMF <= 0.19
There are 1427 non_essential genes that are unused in pairs.
I deleted 1040747 pairs in total.


### DMF_t = 0.5

In [57]:
nodes = nodes[['GENE_id', 'mapped_id', 'type']]
nodes, fd = sample_large_SNL_pairs_with_split_50_and_80(nodes=nodes, pairs=pairs, DMF_t=0.5)

I will not use any interaction value to define SNL pairs.
There are 269203 SL pairs, composed of 3841 individual non_essential genes.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 80/20 split:
I will use 80.0% of the 3841 genes to be used exclusively in the train set.
I will use 3073 genes in the train set and 768 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 163754 SL pairs in the train_set
 - 12787 SL pairs in the test set
 - 92662 SL pairs with one gene in the train set and the other one in the test set.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 50/50 split:
I will use 50.0% of the 3841 genes to be used exclusively in the train set.
I will use 1921 genes in the train set and 1920 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 68852 SL pairs in the train_set
 - 65503 SL pairs in the test 

In [58]:
print(fd[fd['type']=='SL']['mask_05'].value_counts())
print(fd[fd['type']=='SNL']['mask_05'].value_counts())
print(fd[fd['type']=='SL']['mask_08'].value_counts())
print(fd[fd['type']=='SNL']['mask_08'].value_counts())

mask_05
val      134848
train     68852
test      65503
Name: count, dtype: int64
mask_05
val      134848
train     68852
test      65503
Name: count, dtype: int64
mask_08
train    163754
val       92662
test      12787
Name: count, dtype: int64
mask_08
train    163754
val       92662
test      12787
Name: count, dtype: int64


In [59]:
fd.to_csv(
    path_or_buf=data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.5_3841_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=True, index=False, sep='\t', mode='x')

### DMF_t = 0.4

In [60]:
fd = pd.read_csv(
    data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.5_3841_genes_no_doublons_disjoint_sets_splits_05_08.tsv',
    header=0, sep='\t', dtype={3:str, 4:str}
)
pairs = fd[fd['GENE_id'].str.contains('_')]
nodes = nodes[['GENE_id', 'mapped_id', 'type']]
pairs = pairs.rename(columns={'mapped_id_left':'node1_ID', 'mapped_id_right':'node2_ID'})

pairs['type'].value_counts()

type
SL     269203
SNL    269203
Name: count, dtype: int64

In [61]:
nodes, fd = sample_large_SNL_pairs_with_split_50_and_80(nodes=nodes, pairs=pairs, DMF_t=0.4)

I will not use any interaction value to define SNL pairs.
There are 153644 SL pairs, composed of 3841 individual non_essential genes.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 80/20 split:
I will use 80.0% of the 3841 genes to be used exclusively in the train set.
I will use 3073 genes in the train set and 768 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 98159 SL pairs in the train_set
 - 6260 SL pairs in the test set
 - 49225 SL pairs with one gene in the train set and the other one in the test set.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 50/50 split:
I will use 50.0% of the 3841 genes to be used exclusively in the train set.
I will use 1921 genes in the train set and 1920 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 46089 SL pairs in the train_set
 - 30677 SL pairs in the test se

In [62]:
print(fd[fd['type']=='SL']['mask_05'].value_counts())
print(fd[fd['type']=='SNL']['mask_05'].value_counts())
print(fd[fd['type']=='SL']['mask_08'].value_counts())
print(fd[fd['type']=='SNL']['mask_08'].value_counts())

mask_05
val      76878
train    46089
test     30677
Name: count, dtype: int64
mask_05
val      76878
train    46089
test     30677
Name: count, dtype: int64
mask_08
train    98159
val      49225
test      6260
Name: count, dtype: int64
mask_08
train    98159
val      49225
test      6260
Name: count, dtype: int64


In [63]:
fd.to_csv(
    path_or_buf=data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.4_3841_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=True, index=False, sep='\t', mode='x')

### DMF_t = 0.3

In [64]:
fd = pd.read_csv(
    data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.4_3841_genes_no_doublons_disjoint_sets_splits_05_08.tsv',
    header=0, sep='\t', dtype={3:str, 4:str}
)
pairs = fd[fd['GENE_id'].str.contains('_')]
nodes = nodes[['GENE_id', 'mapped_id', 'type']]
pairs = pairs.rename(columns={'mapped_id_left':'node1_ID', 'mapped_id_right':'node2_ID'})

pairs['type'].value_counts()

type
SL     153644
SNL    153644
Name: count, dtype: int64

In [65]:
nodes, fd = sample_large_SNL_pairs_with_split_50_and_80(nodes=nodes, pairs=pairs, DMF_t=0.3)

I will not use any interaction value to define SNL pairs.
There are 68714 SL pairs, composed of 3841 individual non_essential genes.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 80/20 split:
I will use 80.0% of the 3841 genes to be used exclusively in the train set.
I will use 3073 genes in the train set and 768 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 42887 SL pairs in the train_set
 - 3036 SL pairs in the test set
 - 22791 SL pairs with one gene in the train set and the other one in the test set.
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX 
For the 50/50 split:
I will use 50.0% of the 3841 genes to be used exclusively in the train set.
I will use 1921 genes in the train set and 1920 genes in the test set.
With the random sampling of individual genes that I have done, there are:
 - 20344 SL pairs in the train_set
 - 14020 SL pairs in the test set

In [66]:
print(fd[fd['type']=='SL']['mask_05'].value_counts())
print(fd[fd['type']=='SNL']['mask_05'].value_counts())
print(fd[fd['type']=='SL']['mask_08'].value_counts())
print(fd[fd['type']=='SNL']['mask_08'].value_counts())

mask_05
val      34350
train    20344
test     14020
Name: count, dtype: int64
mask_05
val      34350
train    20344
test     14020
Name: count, dtype: int64
mask_08
train    42887
val      22791
test      3036
Name: count, dtype: int64
mask_08
train    42887
val      22791
test      3036
Name: count, dtype: int64


In [67]:
fd.to_csv(
    path_or_buf=data_processed_path+'20250829_yeasts_SL_matched_SNL_no_interaction_DMF_0.3_3841_genes_no_doublons_disjoint_sets_splits_05_08.tsv', 
    header=True, index=False, sep='\t', mode='x')